# ML Meter — Synthetic Dataset Generation

This notebook generates a parallel corpus of **(expanded prose, original poetic line)** pairs for training a GRU Seq2Seq metrical compression model.

**Expansion rule:** The LLM may add, reorder, or rephrase words freely, but **every word from the original line must appear verbatim in the expanded version**.

---
### Pipeline
```
Raw poems → line splitting → LLM expansion → validation → CSV dataset
```

## 0. Install dependencies

In [ ]:
!pip install anthropic pandas tqdm

## 1. Imports & configuration

In [ ]:
import anthropic
import pandas as pd
import json
import time
import re
from pathlib import Path
from tqdm.notebook import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_POEMS_DIR  = Path("data/raw_poems")          # .txt files, one poem each
OUTPUT_DIR     = Path("data/synthetic_pairs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ──────────────────────────────────────────────────────────────────
MODEL          = "claude-haiku-4-5-20251001"     # fast & cheap for bulk generation
MAX_TOKENS     = 256
RETRY_LIMIT    = 3
SLEEP_BETWEEN  = 0.3   # seconds between API calls (rate-limit headroom)

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from environment
print("Client ready.")

## 2. Load raw poems

In [ ]:
def load_poems(poems_dir: Path) -> dict[str, str]:
    """Return {filename_stem: poem_text} for every .txt in poems_dir."""
    poems = {}
    for path in sorted(poems_dir.glob("*.txt")):
        poems[path.stem] = path.read_text(encoding="utf-8")
    print(f"Loaded {len(poems)} poem(s) from {poems_dir}")
    return poems


def split_lines(poem_text: str, min_tokens: int = 3) -> list[str]:
    """Split poem into non-empty lines, filtering out very short fragments."""
    lines = []
    for raw in poem_text.splitlines():
        line = raw.strip()
        if line and len(line.split()) >= min_tokens:
            lines.append(line)
    return lines


# ── Demo with a few hard-coded lines if no files exist yet ─────────────────
DEMO_LINES = [
    "Shall I compare thee to a summer's day",
    "Rough winds do shake the darling buds of May",
    "And summer's lease hath all too short a date",
    "Nor lose possession of that fair thou ow'st",
    "When in eternal lines to time thou grow'st",
]

if RAW_POEMS_DIR.exists():
    poems = load_poems(RAW_POEMS_DIR)
    all_lines = []
    for name, text in poems.items():
        for line in split_lines(text):
            all_lines.append({"poem": name, "original": line})
else:
    print("No raw_poems/ directory found — using demo lines.")
    all_lines = [{"poem": "demo", "original": l} for l in DEMO_LINES]

print(f"Total lines to expand: {len(all_lines)}")
pd.DataFrame(all_lines).head()

## 3. LLM expansion

In [ ]:
SYSTEM_PROMPT = """\
You are a poetic expansion assistant.
Your task: expand a compressed poetic line into natural, flowing prose.

STRICT RULE — every single word from the original line MUST appear verbatim
somewhere in your expansion (order may change).
You may add new words freely to make the prose read naturally.
Do NOT omit any word from the original.

Return ONLY the expanded line — no explanation, no punctuation changes beyond
what is natural, no quotation marks."""

USER_TEMPLATE = "Expand this poetic line:\n{line}"


def expand_line(line: str) -> str | None:
    """Call Claude to expand a single poetic line. Returns expanded string or None on failure."""
    for attempt in range(RETRY_LIMIT):
        try:
            msg = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": USER_TEMPLATE.format(line=line)}],
            )
            return msg.content[0].text.strip()
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)   # exponential back-off
    return None


# Quick smoke-test
test = expand_line("Shall I compare thee to a summer's day")
print("Expansion test:")
print(" ", test)

## 4. Validation — all original words present

In [ ]:
def tokenize(text: str) -> list[str]:
    """Lowercase, strip punctuation, split into tokens."""
    return re.findall(r"[a-z']+", text.lower())


def validate_expansion(original: str, expanded: str) -> tuple[bool, list[str]]:
    """
    Returns (is_valid, missing_words).
    A pair is valid if every token in `original` appears at least once in `expanded`.
    """
    orig_tokens  = tokenize(original)
    exp_tokens   = set(tokenize(expanded))
    missing      = [t for t in orig_tokens if t not in exp_tokens]
    return (len(missing) == 0), missing


# Demo check
ok, missing = validate_expansion("Shall I compare thee to a summer's day", test or "")
print("Valid:", ok, "| Missing words:", missing)

## 5. Generate the full dataset

In [ ]:
records   = []   # valid pairs
failures  = []   # lines that failed validation after retries

for item in tqdm(all_lines, desc="Expanding lines"):
    original = item["original"]
    expanded = None
    valid    = False
    missing  = []

    # Allow a second LLM attempt if validation fails on the first pass
    for _ in range(2):
        expanded = expand_line(original)
        if expanded:
            valid, missing = validate_expansion(original, expanded)
            if valid:
                break
        time.sleep(SLEEP_BETWEEN)

    if valid:
        records.append({
            "poem"     : item["poem"],
            "input"    : expanded,    # model input  (expanded prose)
            "target"   : original,    # model target (compressed poetic line)
        })
    else:
        failures.append({
            "poem"     : item["poem"],
            "original" : original,
            "expanded" : expanded,
            "missing"  : missing,
        })

print(f"\n✅  Valid pairs : {len(records)}")
print(f"❌  Failures    : {len(failures)}")

## 6. Inspect results

In [ ]:
df = pd.DataFrame(records)
df.head(10)

In [ ]:
# Quick length stats
df["input_len"]  = df["input"].str.split().str.len()
df["target_len"] = df["target"].str.split().str.len()
df[["input_len", "target_len"]].describe().round(1)

In [ ]:
# Show any failures for manual review
if failures:
    print("Lines that failed validation:")
    for f in failures:
        print(f"  Original : {f['original']}")
        print(f"  Expanded : {f['expanded']}")
        print(f"  Missing  : {f['missing']}")
        print()
else:
    print("No failures — all lines passed validation.")

## 7. Export

In [ ]:
# ── CSV for training ────────────────────────────────────────────────────────
csv_path = OUTPUT_DIR / "synthetic_pairs.csv"
df[["poem", "input", "target"]].to_csv(csv_path, index=False)
print(f"Saved {len(df)} pairs → {csv_path}")

# ── JSONL (friendlier for HuggingFace / custom loaders) ────────────────────
jsonl_path = OUTPUT_DIR / "synthetic_pairs.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(json.dumps({"input": row["input"], "target": row["target"]}) + "\n")
print(f"Saved JSONL   → {jsonl_path}")

# ── Failure log ─────────────────────────────────────────────────────────────
if failures:
    fail_path = OUTPUT_DIR / "failures.json"
    with open(fail_path, "w", encoding="utf-8") as f:
        json.dump(failures, f, indent=2)
    print(f"Failures log  → {fail_path}")

---
## Notes

| Column | Description |
|--------|-------------|
| `input` | Expanded prose version — fed to the GRU encoder |
| `target` | Original poetic line — the decoder learns to reconstruct this |

**Validation logic:** every token produced by `re.findall(r"[a-z']+", original.lower())` must appear at least once in the expanded string. Contractions like `ow'st` and `thou` are handled correctly.

**Switching models:** change `MODEL` in cell 1. `claude-haiku-4-5-20251001` is recommended for bulk runs; swap to `claude-sonnet-4-6` for higher-quality expansions on difficult archaic lines.

**Next step:** open `baseline_gru.ipynb` and point it at `data/synthetic_pairs/synthetic_pairs.csv`.